In [5]:
import pandas as pd
import numpy as np


In [ ]:
# --- Colab bootstrap (safe to run locally too) ---
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ElNino'   # <-- match your Drive folder
except ImportError:
    BASE_DIR = '.'
os.chdir(BASE_DIR)
print('Working dir:', os.getcwd())
print('SA data present:', os.path.exists('data/era5_southasia_with_precip_1980_2025.csv'))

In [6]:
df=pd.read_csv("data/era5_southasia_with_precip_1980_2025.csv")
df.head(10)

,time,latitude,longitude,sst,t2m,msl,avg_iews,avg_inss,ttr,tp,t2m_anom,t2m_anom_z,tp_anom,tp_anom_z,msl_anom,msl_anom_z,avg_iews_anom,avg_iews_anom_z,avg_inss_anom,avg_inss_anom_z
0,1980-01-01,5.0,60.0,300.76367,299.47586,101179.750,-0.043203,-0.051379,-24779760.0,0.001906,-0.031191,0.052047,-0.001582,-0.774347,-0.472935,0.044969,-0.004818,-0.137634,-0.003910,-0.114570
1,1980-01-01,5.0,62.0,301.22656,299.68290,101167.750,-0.039419,-0.048205,-24774896.0,0.001506,0.006267,0.086922,-0.002289,-1.125598,1.820652,0.063266,-0.004208,-0.123206,-0.007664,-0.216373
2,1980-01-01,5.0,64.0,301.1631,299.81766,101156.375,-0.034292,-0.040576,-24753648.0,0.001185,0.016664,0.096602,-0.002711,-1.334607,1.940435,0.064222,-0.002159,-0.074778,-0.007000,-0.198380
3,1980-01-01,5.0,66.0,301.1631,299.92703,101147.750,-0.030996,-0.035449,-24404208.0,0.001672,0.053259,0.130673,-0.001928,-0.946176,0.894022,0.055874,-0.001350,-0.055646,-0.007559,-0.213530
4,1980-01-01,5.0,68.0,301.46387,299.97586,101140.250,-0.028067,-0.031482,-24153328.0,0.009810,0.026256,0.105533,0.006464,3.218050,-1.029891,0.040526,-0.000320,-0.031306,-0.008984,-0.252178
5,1980-01-01,5.0,70.0,301.39648,299.85280,101137.500,-0.027945,-0.028369,-24398320.0,0.001776,-0.085127,0.001831,-0.000814,-0.393385,-0.828587,0.042132,0.000012,-0.023465,-0.009374,-0.262758
6,1980-01-01,5.0,72.0,301.16992,299.84695,101135.250,-0.027578,-0.025256,-23890416.0,0.001410,-0.063634,0.021841,-0.000927,-0.449650,-1.121957,0.039792,0.004586,0.084682,-0.006362,-0.181071
7,1980-01-01,5.0,74.0,301.5205,300.10672,101128.500,-0.042837,-0.037341,-23890672.0,0.000873,0.134146,0.205983,-0.002084,-1.023531,-3.651848,0.019610,0.002487,0.035062,-0.008132,-0.229075
8,1980-01-01,5.0,76.0,301.34668,300.07547,101117.750,-0.060171,-0.070483,-23361264.0,0.001179,0.194437,0.262116,-0.002497,-1.228775,-7.923804,-0.014468,-0.007084,-0.191205,-0.022377,-0.615415
9,1980-01-01,5.0,78.0,301.5244,299.94460,101101.000,-0.037222,-0.072803,-22490608.0,0.001784,0.163050,0.232893,-0.001945,-0.954674,-15.331848,-0.073564,-0.007135,-0.192397,-0.029092,-0.797538


In [7]:
df.shape

(208656, 20)

In [8]:
df['time'] = pd.to_datetime(df['time'])
print(f"Num Unique Times: {df['time'].nunique()}")
print(f"Lats: {df['latitude'].nunique()}, Lons: {df['longitude'].nunique()}")
print(f"Total per month -> {df['latitude'].nunique() * df['longitude'].nunique()}")

Num Unique Times: 552
Lats: 18, Lons: 21
Total per month -> 378


## Multi-task CNN-TCN — South Asia impact + ENSO (Objective 2)

**Design (confirmed with the team):**
- **Input:** 12-month windows of the **South Asia** grid with five `_z` channels
  (`t2m`, `tp`, `msl`, zonal wind `iews`, meridional wind `inss`). No Pacific fields here.
- **Shared encoder:** the same CNN-TCN as the ENSO notebook produces one encoded vector `z`.
- **Two heads (multi-task):** an **ENSO head** (monthly Niño 3.4 ONI scalar) and an
  **impact head** (single-month anomalies of `t2m` and `tp` over all 18×21 = 378 cells → 756-vector).
- **Loss:** `L_total = alpha * L_ENSO + (1 - alpha) * L_impact` (Eq 4-12).
- **No leakage:** the 12-month input window ends `LEAD` months *before* the target month.
- **Rolling monthly targets:** every calendar month (not just JJAS) is a training sample now,
  expanding N from ~45 (annual JJAS-only) to ~500+ (one sample per month), which gives the
  impact head enough data density to learn real spatial structure instead of blurry averages.
- **Regularization:** dropout raised to `0.3` throughout the shared encoder and both heads to
  fight overfitting now that the impact head has more capacity to memorize.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

# --- Config ---
SA_VARS     = ['t2m_anom_z', 'tp_anom_z', 'msl_anom_z', 'avg_iews_anom_z', 'avg_inss_anom_z']
IMPACT_VARS = ['t2m_anom_z', 'tp_anom_z']   # predicted fields (channels 0 and 1 of SA_VARS)
SEQ_LEN     = 12            # months per input window
LEAD        = 3             # forecast issued this many months before the target month
TARGET_COL  = 'nino34_3m'   # 3-month running-mean Nino 3.4 = official ONI

SA_CSV   = 'data/era5_southasia_with_precip_1980_2025.csv'
NINO_CSV = 'data/Combined/nino34_index_3month_running_mean_official_oni.csv'

BATCH_SIZE   = 8
EPOCHS       = 400
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.3          # raised from 0.1 to fight overfitting
ALPHA        = 0.5          # weight on the ENSO loss in L_total
LAMBDA_CORR  = 0.5          # weight on the (1 - spatial-r) term inside L_impact
PATIENCE     = 60
CKPT_PATH    = 'cnn_tcn_impact_best.pt'

### Build the South Asia spatio-temporal tensor `(T, H, W, C)`

Reindex onto a complete `(time × lat × lon)` grid so ordering is guaranteed, then fill any
missing cell with `0.0` (neutral anomaly).

In [ ]:
df['time'] = pd.to_datetime(df['time'])
times = np.sort(df['time'].unique())
lats  = np.sort(df['latitude'].unique())
lons  = np.sort(df['longitude'].unique())
T, H, W, C = len(times), len(lats), len(lons), len(SA_VARS)
print(f'T={T} months | H={H} lats | W={W} lons | C={C} channels')

times_pd = pd.to_datetime(times)
full_idx = pd.MultiIndex.from_product([times_pd, lats, lons],
                                      names=['time', 'latitude', 'longitude'])
gridded = (df.set_index(['time', 'latitude', 'longitude'])[SA_VARS].reindex(full_idx))
grid = np.full((T, H, W, C), np.nan, dtype=np.float32)
for c, feat in enumerate(SA_VARS):
    grid[..., c] = gridded[feat].to_numpy(dtype=np.float32).reshape(T, H, W)
nan_frac = np.isnan(grid).mean(axis=(0, 1, 2))
grid = np.nan_to_num(grid, nan=0.0)
print('SA grid tensor (T, H, W, C):', grid.shape)
for f, fr in zip(SA_VARS, nan_frac):
    print(f'  {f:<18} NaN filled: {fr*100:5.1f}%')

### Build rolling monthly samples, ENSO target, and impact target

For **every** calendar month `M` in the record: the input window is the 12 months ending
`LEAD` months before `M`; the ENSO target is the ONI value at month `M`; the impact target
is the single-month anomaly of `t2m` and `tp` over every grid cell, flattened as

`[t2m(all cells), tp(all cells)]` (length 756). This turns ~45 annual samples into onesample per valid month (~500+), giving the impact head far more training signal.

In [ ]:
nino = pd.read_csv(NINO_CSV)
nino['time'] = pd.to_datetime(nino['time'])
oni = nino.set_index('time')[TARGET_COL]

month_index = {ts: i for i, ts in enumerate(times_pd)}

X_list, yE_list, yI_list, yrs, mos = [], [], [], [], []
for t_idx, target_ts in enumerate(times_pd):
    win_end = target_ts - pd.DateOffset(months=LEAD)          # last month of input window
    win_ts  = [win_end - pd.DateOffset(months=SEQ_LEN - 1 - k) for k in range(SEQ_LEN)]
    if any(ts not in month_index for ts in win_ts): continue
    oni_val = oni.get(target_ts, np.nan)
    if pd.isna(oni_val): continue

    idxs = [month_index[ts] for ts in win_ts]
    Xw   = grid[idxs]                                          # (12, H, W, C)
    imp  = grid[t_idx][..., :2]                                # (H, W, 2) single-month anomaly
    imp  = np.transpose(imp, (2, 0, 1)).reshape(-1)            # (2*H*W,) order [t2m..., tp...]

    X_list.append(Xw); yE_list.append(float(oni_val)); yI_list.append(imp)
    yrs.append(target_ts.year); mos.append(target_ts.month)

X        = np.stack(X_list).astype(np.float32)                # (N, 12, H, W, C)
y_enso   = np.asarray(yE_list, dtype=np.float32)              # (N,)
y_impact = np.stack(yI_list).astype(np.float32)               # (N, 756)
yrs      = np.asarray(yrs)
mos      = np.asarray(mos)
assert np.isfinite(X).all() and np.isfinite(y_enso).all() and np.isfinite(y_impact).all()
print('X:', X.shape, '| y_enso:', y_enso.shape, '| y_impact:', y_impact.shape)
print('Rolling monthly samples: N =', len(yrs), '| years', yrs.min(), '->', yrs.max())

### Chronological split + per-channel re-standardization (train stats only)

In [ ]:
train_mask = yrs <= 2018
val_mask   = (yrs >= 2019) & (yrs <= 2022)
test_mask  = yrs >= 2023
for n, m in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
    if m.sum():
        print(f'{n:5s}: n={m.sum():3d}  years {yrs[m].min()}-{yrs[m].max()}')
    else:
        print(f'{n:5s}: n=0')

mu = X[train_mask].mean(axis=(0, 1, 2, 3))         # (C,)
sd = X[train_mask].std(axis=(0, 1, 2, 3)) + 1e-8   # (C,)
X_std = (X - mu) / sd
print('Per-channel mean:', np.round(mu, 4))
print('Per-channel std :', np.round(sd, 4))

In [ ]:
def to_t(mask):
    xt = torch.from_numpy(X_std[mask]).permute(0, 1, 4, 2, 3).contiguous()  # (N,T,C,H,W)
    e  = torch.from_numpy(y_enso[mask]).float()
    im = torch.from_numpy(y_impact[mask]).float()
    return xt, e, im

Xtr, Etr, Itr = to_t(train_mask)
Xva, Eva, Iva = to_t(val_mask)
Xte, Ete, Ite = to_t(test_mask)
print('Xtr:', tuple(Xtr.shape), '| Xva:', tuple(Xva.shape), '| Xte:', tuple(Xte.shape))

train_loader = DataLoader(TensorDataset(Xtr, Etr, Itr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Eva, Iva), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(Xte, Ete, Ite), batch_size=BATCH_SIZE)

### Model — shared CNN-TCN encoder + ENSO head + impact head

The spatial CNN and dilated-causal TCN are identical to the ENSO notebook; only a second
(impact) head is added on top of the shared encoded vector `z`.

In [ ]:
class SpatialEncoder(nn.Module):
    """2-D CNN applied independently to each monthly snapshot -> feature vector of dim d."""
    def __init__(self, in_ch, d=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),    nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, d, 3, padding=1),     nn.BatchNorm2d(d),  nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
        )
        self.d = d

    def forward(self, x):                 # x: (B, T, C, H, W)
        B, Tm, Cc, Hh, Ww = x.shape
        x = x.reshape(B * Tm, Cc, Hh, Ww)
        return self.cnn(x).reshape(B, Tm, self.d)

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp): super().__init__(); self.chomp = chomp
    def forward(self, x): return x[:, :, :-self.chomp].contiguous() if self.chomp > 0 else x


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(inplace=True), nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x if self.down is None else self.down(x)
        return self.relu(self.net(x) + res)


class TemporalEncoder(nn.Module):
    def __init__(self, in_dim, channels=(128, 64, 64), kernel_size=3, dropout=0.1):
        super().__init__()
        layers, prev = [], in_dim
        for l, ch in enumerate(channels):
            layers.append(TemporalBlock(prev, ch, kernel_size, dilation=2 ** l, dropout=dropout))
            prev = ch
        self.tcn = nn.Sequential(*layers)
        self.out_dim = prev

    def forward(self, f):                 # f: (B, T, d)
        z = self.tcn(f.transpose(1, 2))   # (B, ch, T)
        return z[:, :, -1]                # last step -> (B, ch)

In [ ]:
class CNN_TCN_MultiTask(nn.Module):
    def __init__(self, in_ch, n_impact, d=128, tcn_channels=(128, 64, 64),
                 head_hidden=64, dropout=0.1):
        super().__init__()
        self.spatial  = SpatialEncoder(in_ch, d)
        self.temporal = TemporalEncoder(d, tcn_channels, dropout=dropout)
        h = self.temporal.out_dim
        self.enso_head = nn.Sequential(
            nn.Linear(h, head_hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )
        self.impact_head = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, n_impact),
        )

    def forward(self, x):
        z = self.temporal(self.spatial(x))
        return self.enso_head(z).squeeze(-1), self.impact_head(z)


model = CNN_TCN_MultiTask(in_ch=C, n_impact=y_impact.shape[1], dropout=DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}  | impact outputs: {y_impact.shape[1]} (= 2 x {H*W})')

### Multi-task training — `L_total = alpha * L_ENSO + (1 - alpha) * L_impact`

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=20)
N_IMPACT_VARS = len(IMPACT_VARS)


def spatial_corr_loss(pred, target, n_vars=N_IMPACT_VARS):
    """1 - per-sample, per-variable Pearson r, averaged (rewards matching the spatial pattern)."""
    B = pred.shape[0]
    p = pred.view(B, n_vars, -1) - pred.view(B, n_vars, -1).mean(dim=-1, keepdim=True)
    t = target.view(B, n_vars, -1) - target.view(B, n_vars, -1).mean(dim=-1, keepdim=True)
    num = (p * t).sum(dim=-1)
    den = torch.sqrt((p ** 2).sum(dim=-1) * (t ** 2).sum(dim=-1) + 1e-8)
    r = num / den
    return (1 - r).mean()


def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    tot, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, eb, ib in loader:
            xb, eb, ib = xb.to(device), eb.to(device), ib.to(device)
            pe, pi = model(xb)
            l_impact = criterion(pi, ib) + LAMBDA_CORR * spatial_corr_loss(pi, ib)
            loss = ALPHA * criterion(pe, eb) + (1 - ALPHA) * l_impact
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * xb.size(0); n += xb.size(0)
    return tot / n


best, best_ep, wait = float('inf'), -1, 0
hist = {'train': [], 'val': []}
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    scheduler.step(va); hist['train'].append(tr); hist['val'].append(va)
    if va < best - 1e-6:
        best, best_ep, wait = va, ep, 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        wait += 1
    if ep % 20 == 0 or ep == 1:
        print(f'epoch {ep:3d}  train={tr:.4f}  val={va:.4f}  (best {best:.4f} @ {best_ep})')
    if wait >= PATIENCE:
        print(f'Early stop at epoch {ep}.'); break

print(f'\nBest val loss={best:.4f} at epoch {best_ep}.')
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist['train'], label='train L_total')
plt.plot(hist['val'],   label='val L_total')
plt.axvline(best_ep - 1, color='r', ls='--', lw=0.8, label=f'best @ {best_ep}')
plt.xlabel('epoch'); plt.ylabel('weighted MSE'); plt.title('Multi-task training curve')
plt.legend(); plt.tight_layout(); plt.show()

### Evaluation — ENSO head and impact head (per variable)

In [ ]:
from scipy.stats import pearsonr


@torch.no_grad()
def predict(xt):
    model.eval(); es, isb = [], []
    for i in range(0, xt.size(0), BATCH_SIZE):
        pe, pi = model(xt[i:i + BATCH_SIZE].to(device))
        es.append(pe.cpu()); isb.append(pi.cpu())
    return torch.cat(es).numpy(), torch.cat(isb).numpy()


def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

pe_te, pi_te = predict(Xte)
print('== ENSO head (monthly ONI) ==')
print('test RMSE:', round(rmse(pe_te, Ete.numpy()), 4))
if len(Ete) > 1:
    print('test Pearson r:', round(pearsonr(pe_te, Ete.numpy())[0], 4))

print('\n== Impact head (monthly anomalies) ==')
pi_grid = pi_te.reshape(-1, 2, H, W)
it_grid = Ite.numpy().reshape(-1, 2, H, W)
for k, name in enumerate(['t2m', 'tp']):
    a, b = pi_grid[:, k].ravel(), it_grid[:, k].ravel()
    line = f'{name:4s}  test RMSE={rmse(a, b):.4f}'
    if a.size > 1:
        line += f'  spatial-r={pearsonr(a, b)[0]:.4f}'
    print(line)

In [ ]:
# --- Predicted vs actual monthly anomaly maps for the last test month ---
yr = -1
year_label  = yrs[test_mask][yr]
month_label = mos[test_mask][yr]
month_name  = pd.Timestamp(year=2000, month=int(month_label), day=1).strftime('%b')
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
extent = [lons.min(), lons.max(), lats.min(), lats.max()]
for k, name in enumerate(['t2m', 'tp']):
    vmax = np.abs(it_grid[yr, k]).max()
    for j, (arr, title) in enumerate([(it_grid[yr, k], 'Actual'), (pi_grid[yr, k], 'Predicted')]):
        ax = axes[k, j]
        im = ax.imshow(arr, origin='lower', extent=extent, cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax, aspect='auto')
        ax.set_title(f'{name} {title}  {month_name} {year_label}')
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
torch.save({'state_dict': model.state_dict(),
            'mu': mu, 'sd': sd,
            'config': {'SA_VARS': SA_VARS, 'IMPACT_VARS': IMPACT_VARS,
                       'SEQ_LEN': SEQ_LEN, 'LEAD': LEAD, 'DROPOUT': DROPOUT,
                       'H': H, 'W': W, 'C': C, 'ALPHA': ALPHA}},
           'cnn_tcn_impact_full.pt')
print('Saved -> cnn_tcn_impact_full.pt  (weights + norm stats + config)')